# Compiler Scheduler - GRPO Training (Llama-3.1-8B)

Train **Llama-3.1-8B** to learn **when to fuse operations** and **when to retain tensors** in fast memory.

Trained via GRPO with reference-anchored DPO.



**Previous experiments:**

- **env1_v1** — Vanilla multi-step GRPO on FusionOps (V1). Learned basic fusion (0.273 mean) but no retention or split-K. [Colab](https://colab.research.google.com/drive/1F9RXon5vpSv8zww-w19ZK4cld1qCjdCz?usp=sharing)
- **env1_v2** — Added hint-guided exploration. Unlocked split-K and hit theoretical ceiling on linear chain (0.669). Mean: 0.405. [Colab](https://colab.research.google.com/drive/1nHo8L4jy9s3CfC4guZfXJgaJZzaSoVs6?usp=sharing)
- **env1_v3** — Added 2-step lookahead. Retention breakthrough on diamond (0.273→0.553, beat env reference). Mean: 0.475. But generalization collapsed to 0.049 — model memorized, didn't learn. **Led to full env rebuild.** [Colab](https://colab.research.google.com/drive/1eWMkfeAGTgFkSz6JoJJRDPuP1FIJTG1o?usp=sharing)
- **env2_v1** — New env (Compiler-Scheduler), sequential actions, 5 tasks up to 24 ops, greedy baseline. Generalization gap 10x→2.3x. Fusion hit 100%, hints beaten to 0%. But retention stuck at heuristic level. [Colab](https://colab.research.google.com/drive/1jbUlkI9_Lmn4yidw-udDKT4LhrphS30U?usp=sharing)


**This run:** Llama-3.1-8B (4-bit, LoRA r=32), 500 episodes on V2. Testing whether 8B capacity breaks the retention plateau.

In [ ]:
# Cell 1: System check
import subprocess, shutil, sys
print("Python:", sys.version.split()[0])
print("\n=== GPU ===")
try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv"]).decode())
except Exception as e:
    print("nvidia-smi not available:", e)
print("=== Disk ===")
for p in ["/workspace", "/root"]:
    try:
        total, used, free = shutil.disk_usage(p)
        print(f"{p}: {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")
    except: pass

Python: 3.12.3

=== GPU ===
name, memory.total [MiB], memory.free [MiB], driver_version
NVIDIA GeForce RTX 5090, 32607 MiB, 32109 MiB, 580.126.09

=== Disk ===
/workspace: 1320940.9 GB total, 267135.4 GB free
/root: 53.7 GB total, 49.5 GB free


In [ ]:
# Cell 2: Install dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "typing_extensions>=4.12.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "numpy"], check=True)

import importlib, typing_extensions
importlib.reload(typing_extensions)
from importlib.metadata import version as _v
print(f'Dependencies installed. typing_extensions={_v("typing_extensions")}')

import torch, transformers, trl, peft
print(f'torch={torch.__version__} transformers={transformers.__version__} trl={trl.__version__} peft={peft.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


Dependencies installed. typing_extensions=4.15.0


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


torch=2.10.0+cu128 transformers=5.5.0 trl=0.24.0 peft=0.19.1
CUDA: True
GPU: NVIDIA GeForce RTX 5090


In [ ]:
# Cell 3: Workspace setup
import os, pathlib

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
HF_CACHE = os.path.join(WORKSPACE, "hf_cache")
OUTPUT_DIR = os.path.join(WORKSPACE, "scheduler-training-output")
os.makedirs(HF_CACHE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE, "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"Workspace: {WORKSPACE}\nOutput: {OUTPUT_DIR}")

Workspace: /workspace
Output: /workspace/scheduler-training-output


In [ ]:
# Cell 4: Clone the env repo
REPO_URL = "https://github.com/AnishaRoy5555/compiler-scheduler-env.git"
REPO_DIR = os.path.join(WORKSPACE, "compiler-scheduler-env")

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL}")
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print(f"Pulling latest")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"Working dir: {os.getcwd()}")

Pulling latest
Already up to date.
Working dir: /workspace/compiler-scheduler-env


In [ ]:
# Cell 5: Imports and seed
from unsloth import FastLanguageModel

import json, math, random, re, time, warnings, logging
from copy import deepcopy
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*max_new_tokens.*")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Seed: {SEED}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_1494/286305551.py:2: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Seed: 42


In [ ]:
# Cell 6: Training config (Llama-3.1-8B)
CONFIG = {
  "episodes": 400,
  "eval_interval": 50,
  "num_candidates": 4,
  "lr": 5e-06,
  "grad_accum": 8,
  "model": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
  "seed": 42,
  "max_seq_length": 4096,
  "lora_r": 32,
  "lora_alpha": 64,
  "dpo_beta": 0.1,
  "curriculum_start": 0.2,
  "curriculum_end": 0.8,
  "max_pairs_per_episode": 12
}
CONFIG["output_dir"] = OUTPUT_DIR
CONFIG["seed"] = SEED
print(json.dumps(CONFIG, indent=2))


{
  "episodes": 400,
  "eval_interval": 50,
  "num_candidates": 4,
  "lr": 5e-06,
  "grad_accum": 8,
  "model": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
  "seed": 42,
  "max_seq_length": 4096,
  "lora_r": 32,
  "lora_alpha": 64,
  "dpo_beta": 0.1,
  "curriculum_start": 0.2,
  "curriculum_end": 0.8,
  "max_pairs_per_episode": 12,
  "output_dir": "/workspace/scheduler-training-output"
}


In [ ]:
# Cell 7: Load environment
from src.environment import FusionOpsEnv, parse_action
from src.graph_gen import load_task, list_tasks, generate_training_graph, TASKS
from src.models import Action, FusionGroup
from src.cost_model import compute_greedy_baseline

print("Tasks:", list_tasks())
for name in list_tasks():
    g, cfg = load_task(name)
    env = FusionOpsEnv(g, max_steps=cfg["max_steps"])
    print(f"  {name}: {len(g.nodes)} nodes, greedy={env.greedy_latency:.0f}")

# Test random generation
for level in [0.1, 0.5, 0.9]:
    g = generate_training_graph(level)
    print(f"  curriculum={level}: {len(g.nodes)} nodes")

Tasks: ['task1_chain', 'task2_residual', 'task3_attention', 'task4_mixed', 'task5_adversarial']
  task1_chain: 8 nodes, greedy=118360
  task2_residual: 12 nodes, greedy=2026577
  task3_attention: 16 nodes, greedy=4138689
  task4_mixed: 24 nodes, greedy=3917982
  task5_adversarial: 20 nodes, greedy=2323274
  curriculum=0.1: 16 nodes
  curriculum=0.5: 25 nodes
  curriculum=0.9: 46 nodes


In [ ]:
# Cell 8: Load model (Llama-3.1-8B)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.generation_config.max_length = None
model.generation_config.max_new_tokens = None

FLM = FastLanguageModel
DEVICE = next(model.parameters()).device

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Model: Llama-3.1-8B")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print(f"Device: {DEVICE}")
print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated")


==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model: Llama-3.1-8B
Trainable: 83,886,080 / 4,624,486,400 (1.8%)
Device: cuda:0
GPU Memory: 6.1 GB allocated


In [ ]:
# Cell 9: System prompt and task lists
#
# The system prompt teaches the LLM the V2 action format.
# Key difference from V1: actions are JSON, not SCHEDULE commands.
# The observation is compact JSON with future_uses for retention reasoning.

SYSTEM_PROMPT = """You schedule ML computation graph operations to minimize latency.

You receive a JSON observation with:
- nodes: the graph (step 0 only)
- current_node: the node to schedule now
- current_group: ops in the current fusion kernel
- fast_mem: node IDs in fast memory
- capacity: fast memory size in bytes
- max_fusion: max ops per kernel
- future_uses: how many downstream ops need each node's output
- lookahead: next 2 nodes

Respond with ONLY a JSON object:
{"fuse_with_prev": bool, "tile": int, "retain": [int]}

Rules:
- Fuse consecutive connected ops to eliminate memory transfers
- Retain outputs with future_uses > 0 that will be needed later
- Don't retain dead tensors (future_uses = 0)
- Use tile 128 for pointwise, smaller if memory is tight with fusion
- Don't fuse if it exceeds capacity or max_fusion"""

TRAIN_TASKS = ["task1_chain", "task2_residual", "task3_attention", "task4_mixed", "task5_adversarial"]
EVAL_TASKS = ["task1_chain", "task2_residual", "task3_attention", "task4_mixed", "task5_adversarial"]

print(f"Train: {TRAIN_TASKS}")
print(f"Eval:  {EVAL_TASKS}")

Train: ['task1_chain', 'task2_residual', 'task3_attention', 'task4_mixed', 'task5_adversarial']
Eval:  ['task1_chain', 'task2_residual', 'task3_attention', 'task4_mixed', 'task5_adversarial']


In [ ]:
# Cell 10: Action generation helpers

def generate_action(model, tokenizer, observation, temperature=0.7):
    """Generate a single JSON action from the model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": observation},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=80,
            temperature=max(temperature, 0.01),
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    # Strip markdown fences
    if text.startswith("```"):
        text = "\n".join(l for l in text.split("\n") if not l.startswith("```")).strip()
    # Validate JSON
    try:
        json.loads(text)
        return text
    except:
        return '{"fuse_with_prev": false, "tile": 128, "retain": []}'


def generate_candidates(model, tokenizer, observation, num_candidates=4, temperature=0.8):
    """Generate multiple candidate actions for GRPO scoring."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": observation},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=80,
            temperature=max(temperature, 0.01),
            do_sample=True,
            num_return_sequences=num_candidates,
            pad_token_id=tokenizer.eos_token_id,
        )
    candidates = []
    for seq in outputs:
        gen = seq[inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(gen, skip_special_tokens=True).strip()
        if text.startswith("```"):
            text = "\n".join(l for l in text.split("\n") if not l.startswith("```")).strip()
        try:
            json.loads(text)
            candidates.append(text)
        except:
            candidates.append('{"fuse_with_prev": false, "tile": 128, "retain": []}')
    return candidates


def make_heuristic_action(obs_dict):
    """Generate a heuristic action as a training hint.
    This is the 'oracle' the model must learn to match and eventually beat."""
    current = obs_dict.get("current_node", -1)
    group = obs_dict.get("current_group", {})
    group_ids = group.get("node_ids", [])
    future = obs_dict.get("future_uses", {})
    capacity = obs_dict.get("capacity", 0)
    max_fuse = obs_dict.get("max_fusion", 6)
    fast_mem = obs_dict.get("fast_mem", [])

    # Get current node info
    node_info = obs_dict.get("current_node_info", {})
    if not node_info and "nodes" in obs_dict:
        for n in obs_dict["nodes"]:
            if n["id"] == current:
                node_info = n
                break

    inputs = node_info.get("inputs", [])
    op = node_info.get("op", "relu")

    # Fuse if connected and within limits
    fuse = False
    if group_ids and len(group_ids) < max_fuse:
        connected = any(inp in group_ids for inp in inputs)
        if connected:
            fuse = True

    # Tile: 128 default, 64 for matmul under pressure
    tile = 128
    if op in ("matmul", "conv2d") and len(fast_mem) >= 2:
        tile = 64

    # Retain: keep current node output if it has future uses
    retain = []
    if str(current) in future and future[str(current)] > 0:
        retain.append(current)

    return json.dumps({"fuse_with_prev": fuse, "tile": tile, "retain": retain})


def get_temperature(episode, total):
    progress = episode / total
    if progress < 0.33: return 0.9
    elif progress < 0.66: return 0.7
    else: return 0.5

print("Generation helpers ready.")

Generation helpers ready.


In [ ]:
# Cell 11: Multi-step rollout with candidate scoring

def rollout_episode(graph, env, model, tokenizer, temperature=0.7,
                    num_candidates=4, hint_ratio=0.5):
    """
    Run one episode. At each step:
    1. Generate model candidates + heuristic hint
    2. Score each by actually stepping a cloned env
    3. Pick the best, collect training pairs

    Returns: (score, training_pairs, trajectory_info)
    """
    result = env.reset()
    observation = result.observation

    training_pairs = []  # (obs, candidates, rewards, hint_won, best_action)
    total_reward = 0.0
    steps = 0
    hint_wins = 0
    total_decisions = 0
    fuse_count = 0
    retain_count = 0

    FLM.for_inference(model)

    while not result.done:
        obs_dict = json.loads(observation)

        # Generate candidates: mix of model + heuristic
        num_hint = max(1, int(num_candidates * hint_ratio))
        num_model = num_candidates - num_hint

        candidates = []
        if num_model > 0:
            candidates = generate_candidates(model, tokenizer, observation, num_model, temperature)

        # Add heuristic hints
        hint = make_heuristic_action(obs_dict)
        candidates.append(hint)
        # Add variation of hint (different tile or retain)
        hint_data = json.loads(hint)
        hint_var = dict(hint_data)
        hint_var["tile"] = 64 if hint_data["tile"] == 128 else 128
        candidates.append(json.dumps(hint_var))

        candidates = candidates[:num_candidates]

        # Score each candidate by stepping a cloned env
        step_rewards = []
        for cand in candidates:
            action = parse_action(cand)
            if action is None:
                step_rewards.append(-0.1)
                continue

            # Clone env state and try this action
            test_env = FusionOpsEnv(graph, max_steps=env.max_steps)
            test_env.state = deepcopy(env.state)
            test_env.topo_order = list(env.topo_order)
            test_env.greedy_latency = env.greedy_latency
            test_env.naive_latency = env.naive_latency
            test_result = test_env.step(action)

            # 2-step lookahead: also score the next step
            lookahead_reward = 0.0
            if not test_result.done:
                la_obs = json.loads(test_result.observation)
                la_hint = make_heuristic_action(la_obs)
                la_action = parse_action(la_hint)
                if la_action:
                    la_env = FusionOpsEnv(graph, max_steps=env.max_steps)
                    la_env.state = deepcopy(test_env.state)
                    la_env.topo_order = list(env.topo_order)
                    la_env.greedy_latency = env.greedy_latency
                    la_env.naive_latency = env.naive_latency
                    la_result = la_env.step(la_action)
                    lookahead_reward = la_result.reward

            step_rewards.append(test_result.reward + 0.5 * lookahead_reward)

        # Pick best candidate
        best_idx = max(range(len(step_rewards)), key=lambda i: step_rewards[i])
        best_action_text = candidates[best_idx]
        hint_won = best_idx >= num_model  # hint was better
        if hint_won:
            hint_wins += 1
        total_decisions += 1

        # Track fusion/retention behavior
        best_data = json.loads(best_action_text)
        if best_data.get("fuse_with_prev", False):
            fuse_count += 1
        if best_data.get("retain", []):
            retain_count += 1

        # Collect training pair
        training_pairs.append((observation, candidates, step_rewards, hint_won, best_action_text))

        # Actually step the env with the best action
        action = parse_action(best_action_text)
        result = env.step(action)
        observation = result.observation
        total_reward += result.reward
        steps += 1

    score = env.get_score()
    hint_win_rate = hint_wins / max(total_decisions, 1)
    fuse_rate = fuse_count / max(steps, 1)
    retain_rate = retain_count / max(steps, 1)

    return score, training_pairs, {
        "steps": steps,
        "total_reward": total_reward,
        "hint_win_rate": hint_win_rate,
        "fuse_rate": fuse_rate,
        "retain_rate": retain_rate,
        "latency": env.state.total_latency,
        "greedy": env.greedy_latency,
    }

print("Rollout ready.")

Rollout ready.


In [ ]:
# Cell 12: GRPO training step with reference-anchored DPO

def compute_logprob(model, tokenizer, observation, action_text):
    """Compute log probability of action given observation."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": observation[:2500]},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + action_text

    prompt_ids = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, return_tensors="pt", add_special_tokens=False,
                         truncation=True, max_length=3500)["input_ids"].to(DEVICE)

    prompt_len = prompt_ids.shape[1]
    if prompt_len >= full_ids.shape[1]:
        return (full_ids.sum() * 0.0).float()

    outputs = model(input_ids=full_ids)
    logits = outputs.logits
    shift_logits = logits[:, prompt_len - 1:-1, :]
    shift_labels = full_ids[:, prompt_len:]

    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    token_logp = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)
    return token_logp.sum()


def compute_logprob_ref(model, tokenizer, observation, action_text):
    """Compute reference (frozen base model) log probability."""
    with torch.no_grad(), model.disable_adapter():
        return compute_logprob(model, tokenizer, observation, action_text).detach()


DPO_BETA = CONFIG["dpo_beta"]

def grpo_step(model, tokenizer, observation, candidates, scores, hint_won=False, best_action=None):
    """
    GRPO update: pairwise DPO loss + imitation loss when hint wins.

    DPO loss teaches: prefer higher-scoring actions.
    Imitation loss teaches: when the heuristic wins, learn to GENERATE it.
    Reference anchoring prevents policy collapse.
    """
    if len(candidates) < 2 or len(scores) < 2:
        return 0.0

    mean_score = sum(scores) / len(scores)
    advantages = [s - mean_score for s in scores]

    # Build pairwise comparisons
    pairs = []
    for i in range(len(candidates)):
        for j in range(len(candidates)):
            if advantages[i] > advantages[j] + 0.001:
                pairs.append((candidates[i], candidates[j], advantages[i] - advantages[j]))

    if not pairs and not hint_won:
        return 0.0

    FLM.for_training(model)
    model.train()

    total_loss = torch.tensor(0.0, device=DEVICE, requires_grad=True)

    # 1. Pairwise DPO loss
    if pairs:
        if len(pairs) > 4:
            pairs = sorted(pairs, key=lambda x: -x[2])[:4]

        for better, worse, margin in pairs:
            logp_w = compute_logprob(model, tokenizer, observation, better)
            logp_l = compute_logprob(model, tokenizer, observation, worse)
            logp_w_ref = compute_logprob_ref(model, tokenizer, observation, better)
            logp_l_ref = compute_logprob_ref(model, tokenizer, observation, worse)

            # Reference-anchored DPO (prevents policy collapse)
            diff = DPO_BETA * ((logp_w - logp_w_ref) - (logp_l - logp_l_ref))
            pair_loss = -torch.nn.functional.logsigmoid(diff)
            total_loss = total_loss + pair_loss * min(margin, 1.0)

    # 2. Imitation loss when hint wins (learn to generate good actions)
    if hint_won and best_action:
        logp_hint = compute_logprob(model, tokenizer, observation, best_action)
        imitation_loss = -logp_hint * 0.1  # small weight
        total_loss = total_loss + imitation_loss

    if total_loss.requires_grad:
        total_loss.backward()

    # Memory cleanup for large models
    torch.cuda.empty_cache()

    return total_loss.item()

print("GRPO step ready.")

GRPO step ready.


In [ ]:
# Cell 13: Evaluation

def evaluate_task(task_name, model, tokenizer, num_runs=3):
    scores = []
    for _ in range(num_runs):
        graph, cfg = load_task(task_name)
        env = FusionOpsEnv(graph, max_steps=cfg["max_steps"])
        result = env.reset()
        obs = result.observation

        FLM.for_inference(model)
        while not result.done:
            action_text = generate_action(model, tokenizer, obs, temperature=0.3)
            action = parse_action(action_text)
            if action is None:
                action = Action(fuse_with_prev=False, tile=128, retain=[])
            result = env.step(action)
            obs = result.observation
        scores.append(env.get_score())
    return sum(scores) / len(scores)


def evaluate_generalization(model, tokenizer, num_graphs=5, num_runs=2):
    """Test on random unseen graphs to measure generalization."""
    scores = []
    for seed in range(2000, 2000 + num_graphs):
        for _ in range(num_runs):
            graph = generate_training_graph(0.6)  # medium-hard
            env = FusionOpsEnv(graph)
            result = env.reset()
            obs = result.observation
            FLM.for_inference(model)
            while not result.done:
                action_text = generate_action(model, tokenizer, obs, temperature=0.3)
                action = parse_action(action_text)
                if action is None:
                    action = Action(fuse_with_prev=False, tile=128, retain=[])
                result = env.step(action)
                obs = result.observation
            scores.append(env.get_score())
    return sum(scores) / len(scores)


def evaluate_all(model, tokenizer):
    results = {}
    for task in EVAL_TASKS:
        results[task] = evaluate_task(task, model, tokenizer)
    results["generalization"] = evaluate_generalization(model, tokenizer)
    return results

print("Evaluation ready.")

Evaluation ready.


In [ ]:
# Cell 14: Baseline evaluation (before training)
print("--- Baseline Evaluation ---")
baseline_scores = evaluate_all(model, tokenizer)
for k, v in sorted(baseline_scores.items()):
    print(f"  {k}: {v:.3f}")

--- Baseline Evaluation ---
  generalization: -0.595
  task1_chain: 0.084
  task2_residual: -0.025
  task3_attention: 0.019
  task4_mixed: -0.485
  task5_adversarial: -0.726


In [ ]:
# Cell 15: Main training loop

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG["lr"],
)

training_log = {
    "config": CONFIG,
    "baseline_scores": baseline_scores,
    "episodes": [],
    "eval_history": [],
}

episode_scores = []
episode_losses = []
episode_fuse_rates = []
episode_retain_rates = []
episode_hint_rates = []
per_task_history = {}

print(f"--- Training: {CONFIG['episodes']} episodes ---")
start_time = time.time()

for episode in range(1, CONFIG["episodes"] + 1):
  try:
    progress = episode / CONFIG["episodes"]
    temperature = get_temperature(episode, CONFIG["episodes"])

    # Curriculum: increase difficulty over time
    curriculum = CONFIG["curriculum_start"] + progress * (CONFIG["curriculum_end"] - CONFIG["curriculum_start"])

    # Hint ratio: start with 50% hints, decay to 10%
    hint_ratio = max(0.1, 0.5 * (1 - progress))

    # Pick task: 40% fixed tasks, 60% random curriculum graphs
    if random.random() < 0.4:
        task_name = random.choice(TRAIN_TASKS)
        graph, cfg = load_task(task_name)
        max_steps = cfg["max_steps"]
    else:
        task_name = f"curriculum_{curriculum:.2f}"
        graph = generate_training_graph(curriculum)
        max_steps = len(graph.nodes)

    env = FusionOpsEnv(graph, max_steps=max_steps)

    # Rollout
    score, training_pairs, traj = rollout_episode(
        graph, env, model, tokenizer,
        temperature=temperature,
        num_candidates=CONFIG["num_candidates"],
        hint_ratio=hint_ratio,
    )

    # Limit training pairs for memory safety
    max_p = CONFIG.get("max_pairs_per_episode", len(training_pairs))
    if len(training_pairs) > max_p:
        training_pairs = sorted(training_pairs, key=lambda x: max(x[2])-min(x[2]), reverse=True)[:max_p]

    # GRPO update on collected pairs
    total_loss = 0.0
    for obs, candidates, rewards, hint_won, best_act in training_pairs:
        loss = grpo_step(model, tokenizer, obs, candidates, rewards,
                         hint_won=hint_won, best_action=best_act if hint_won else None)
        total_loss += loss

    avg_loss = total_loss / max(len(training_pairs), 1)

    # Gradient step
    if episode % CONFIG["grad_accum"] == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    # Track metrics
    episode_scores.append(score)
    episode_losses.append(avg_loss)
    episode_fuse_rates.append(traj["fuse_rate"])
    episode_retain_rates.append(traj["retain_rate"])
    episode_hint_rates.append(traj["hint_win_rate"])

    training_log["episodes"].append({
        "ep": episode, "task": task_name, "score": score,
        "loss": avg_loss, "fuse": traj["fuse_rate"],
        "retain": traj["retain_rate"], "hint_wr": traj["hint_win_rate"],
        "latency": traj["latency"], "greedy": traj["greedy"],
    })

    # Log
    if episode % 10 == 0:
        recent = episode_scores[-20:]
        avg_score = sum(recent) / len(recent)
        avg_hint = sum(episode_hint_rates[-20:]) / len(episode_hint_rates[-20:])
        avg_fuse = sum(episode_fuse_rates[-20:]) / len(episode_fuse_rates[-20:])
        avg_ret = sum(episode_retain_rates[-20:]) / len(episode_retain_rates[-20:])
        elapsed = time.time() - start_time
        print(f"Ep {episode:>4d} | score={avg_score:+.3f} | loss={avg_loss:.4f} | "
              f"fuse={avg_fuse:.0%} | retain={avg_ret:.0%} | hint_wr={avg_hint:.0%} | "
              f"{elapsed:.0f}s | {torch.cuda.memory_allocated()/1e9:.1f}GB")

    # Periodic evaluation
    if episode % CONFIG["eval_interval"] == 0:
        eval_scores = evaluate_all(model, tokenizer)
        training_log["eval_history"].append({"ep": episode, **eval_scores})

        # Track per-task
        for k, v in eval_scores.items():
            if k not in per_task_history:
                per_task_history[k] = []
            per_task_history[k].append((episode, v))

        print(f"  EVAL: " + " | ".join(f"{k}={v:.3f}" for k, v in sorted(eval_scores.items())))

        # Save checkpoint
        ckpt_path = os.path.join(CONFIG["output_dir"], f"checkpoint-{episode}")
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)

        # Save partial log (crash resilience)
        with open(os.path.join(CONFIG["output_dir"], "training_log_partial.json"), "w") as f:
            json.dump(training_log, f, indent=2)

  except torch.cuda.OutOfMemoryError:
    print(f"  OOM at episode {episode}, clearing cache...")
    torch.cuda.empty_cache()
    optimizer.zero_grad()
    continue
  except Exception as e:
    print(f"  Error at episode {episode}: {e}")
    continue

total_time = time.time() - start_time
print(f"\nDone. {CONFIG['episodes']} episodes in {total_time:.0f}s ({total_time/3600:.1f}h)")

--- Training: 400 episodes ---


`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Ep   10 | score=-0.039 | loss=0.2658 | fuse=96% | retain=89% | hint_wr=58% | 309s | 7.3GB
Ep   20 | score=+0.023 | loss=0.2885 | fuse=95% | retain=88% | hint_wr=65% | 561s | 7.3GB
Ep   30 | score=+0.084 | loss=0.2595 | fuse=96% | retain=88% | hint_wr=67% | 808s | 7.3GB
Ep   40 | score=+0.084 | loss=0.2347 | fuse=99% | retain=88% | hint_wr=61% | 1053s | 6.9GB
Ep   50 | score=+0.084 | loss=0.1163 | fuse=98% | retain=88% | hint_wr=49% | 1289s | 7.3GB
  EVAL: generalization=-0.298 | task1_chain=0.084 | task2_residual=0.015 | task3_attention=0.019 | task4_mixed=-0.316 | task5_adversarial=-0.336


Unsloth: Restored added_tokens_decoder metadata in /workspace/scheduler-training-output/checkpoint-50/tokenizer_config.json.


Ep   60 | score=+0.024 | loss=0.1226 | fuse=95% | retain=88% | hint_wr=41% | 1884s | 7.3GB
Ep   70 | score=+0.024 | loss=0.0972 | fuse=96% | retain=88% | hint_wr=36% | 2100s | 7.3GB
Ep   80 | score=+0.084 | loss=0.1055 | fuse=98% | retain=88% | hint_wr=28% | 2298s | 6.9GB
Ep   90 | score=+0.084 | loss=0.0634 | fuse=99% | retain=88% | hint_wr=22% | 2478s | 7.3GB
Ep  100 | score=+0.084 | loss=0.0425 | fuse=100% | retain=88% | hint_wr=14% | 2650s | 7.3GB
  EVAL: generalization=0.032 | task1_chain=0.084 | task2_residual=0.015 | task3_attention=0.024 | task4_mixed=0.406 | task5_adversarial=0.128


...

Ep  310 | score=+0.128 | loss=0.0116 | fuse=92% | retain=91% | hint_wr=11% | 8497s | 7.3GB
Ep  320 | score=+0.128 | loss=0.0114 | fuse=92% | retain=91% | hint_wr=9% | 8610s | 6.9GB
Ep  330 | score=+0.084 | loss=0.0223 | fuse=100% | retain=88% | hint_wr=1% | 8726s | 7.3GB
Ep  340 | score=+0.084 | loss=0.0034 | fuse=100% | retain=88% | hint_wr=1% | 8841s | 7.3GB
Ep  350 | score=+0.084 | loss=0.0072 | fuse=100% | retain=88% | hint_wr=0% | 8950s | 7.3GB
  EVAL: generalization=0.175 | task1_chain=0.084 | task2_residual=0.015 | task3_attention=0.034 | task4_mixed=0.527 | task5_adversarial=0.136


Unsloth: Restored added_tokens_decoder metadata in /workspace/scheduler-training-output/checkpoint-350/tokenizer_config.json.


Ep  360 | score=+0.048 | loss=0.0062 | fuse=96% | retain=90% | hint_wr=4% | 9647s | 6.9GB
Ep  370 | score=+0.048 | loss=0.0064 | fuse=96% | retain=90% | hint_wr=5% | 9755s | 7.3GB
Ep  380 | score=+0.084 | loss=0.0021 | fuse=100% | retain=88% | hint_wr=1% | 9849s | 7.3GB
Ep  390 | score=+0.084 | loss=0.0017 | fuse=100% | retain=88% | hint_wr=0% | 9946s | 7.3GB
Ep  400 | score=+0.084 | loss=0.0015 | fuse=100% | retain=88% | hint_wr=0% | 10040s | 6.9GB
  EVAL: generalization=0.258 | task1_chain=0.084 | task2_residual=-0.004 | task3_attention=0.034 | task4_mixed=0.571 | task5_adversarial=0.085


Unsloth: Restored added_tokens_decoder metadata in /workspace/scheduler-training-output/checkpoint-400/tokenizer_config.json.



Done. 400 episodes in 10368s (2.9h)


In [ ]:
# Cell 16: Final evaluation
print("--- Final Evaluation ---")
final_scores = evaluate_all(model, tokenizer)

print(f"\n{'Task':<25s} {'Before':>8s} {'After':>8s} {'Change':>8s}")
print("-" * 50)
for k in sorted(final_scores.keys()):
    before = baseline_scores.get(k, 0.0)
    after = final_scores[k]
    change = after - before
    print(f"{k:<25s} {before:>8.3f} {after:>8.3f} {change:>+8.3f}")

--- Final Evaluation ---

Task                        Before    After   Change
--------------------------------------------------
generalization              -0.595    0.252   +0.847
task1_chain                  0.084    0.084   +0.000
task2_residual              -0.025    0.015   +0.040
task3_attention              0.019    0.034   +0.014
task4_mixed                 -0.485    0.573   +1.058
task5_adversarial           -0.726    0.131   +0.857


In [ ]:
# Cell 17: Save model and results
print(f"Saving to {CONFIG['output_dir']}")

# LoRA adapter
lora_path = os.path.join(CONFIG["output_dir"], "scheduler-lora")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"  Saved: scheduler-lora/")

# Scores
scores_data = {
    "baseline": baseline_scores,
    "final": final_scores,
    "improvement": {k: final_scores[k] - baseline_scores.get(k, 0) for k in final_scores},
}
with open(os.path.join(CONFIG["output_dir"], "scores.json"), "w") as f:
    json.dump(scores_data, f, indent=2)

# Training log
with open(os.path.join(CONFIG["output_dir"], "training_log.json"), "w") as f:
    json.dump(training_log, f, indent=2)

print("  All saved.")

Saving to /workspace/scheduler-training-output


Unsloth: Restored added_tokens_decoder metadata in /workspace/scheduler-training-output/scheduler-lora/tokenizer_config.json.


  Saved: scheduler-lora/
  All saved.


In [ ]:
# Cell 18: Training plots

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Compiler Scheduler GRPO Training", fontsize=14, fontweight="bold")

# Score curve
ax = axes[0, 0]
ax.plot(episode_scores, alpha=0.15, color="blue")
if len(episode_scores) >= 20:
    sm = [sum(episode_scores[max(0,i-20):i+1])/min(i+1,20) for i in range(len(episode_scores))]
    ax.plot(sm, color="blue", linewidth=2)
ax.set_xlabel("Episode"); ax.set_ylabel("Score"); ax.set_title("Score"); ax.grid(True, alpha=0.3)

# Fusion rate
ax = axes[0, 1]
if len(episode_fuse_rates) >= 20:
    sm = [sum(episode_fuse_rates[max(0,i-20):i+1])/min(i+1,20) for i in range(len(episode_fuse_rates))]
    ax.plot(sm, color="green", linewidth=2)
ax.set_xlabel("Episode"); ax.set_ylabel("Fuse Rate"); ax.set_title("Fusion Rate"); ax.grid(True, alpha=0.3)

# Retention rate
ax = axes[0, 2]
if len(episode_retain_rates) >= 20:
    sm = [sum(episode_retain_rates[max(0,i-20):i+1])/min(i+1,20) for i in range(len(episode_retain_rates))]
    ax.plot(sm, color="orange", linewidth=2)
ax.set_xlabel("Episode"); ax.set_ylabel("Retain Rate"); ax.set_title("Retention Rate"); ax.grid(True, alpha=0.3)

# Hint win rate (should decrease as model learns)
ax = axes[1, 0]
if len(episode_hint_rates) >= 20:
    sm = [sum(episode_hint_rates[max(0,i-20):i+1])/min(i+1,20) for i in range(len(episode_hint_rates))]
    ax.plot(sm, color="red", linewidth=2)
ax.set_xlabel("Episode"); ax.set_ylabel("Hint Win Rate"); ax.set_title("Hint Win Rate (lower = model learning)"); ax.grid(True, alpha=0.3)

# Per-task eval
ax = axes[1, 1]
for k in sorted(per_task_history.keys()):
    if per_task_history[k]:
        eps, sc = zip(*per_task_history[k])
        ax.plot(eps, sc, marker="o", label=k[:12], markersize=4)
ax.set_xlabel("Episode"); ax.set_ylabel("Score"); ax.set_title("Per-Task Eval"); ax.legend(fontsize=6); ax.grid(True, alpha=0.3)

# Before vs After
ax = axes[1, 2]
tasks = sorted([t for t in final_scores if t != "generalization"])
x = range(len(tasks))
w = 0.35
bv = [baseline_scores.get(t, 0) for t in tasks]
av = [final_scores.get(t, 0) for t in tasks]
ax.bar([i-w/2 for i in x], bv, w, label="Before", color="lightcoral")
ax.bar([i+w/2 for i in x], av, w, label="After", color="steelblue")
ax.set_xticks(list(x))
ax.set_xticklabels([t.replace("task","T") for t in tasks], fontsize=6, rotation=30)
ax.set_ylabel("Score"); ax.set_title("Before vs After"); ax.legend(); ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plot_path = os.path.join(CONFIG["output_dir"], "training_plots.png")
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved: {plot_path}")

Saved: /workspace/scheduler-training-output/training_plots.png


In [ ]:
# Cell 19: Summary
print("=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)
print(f"Episodes: {CONFIG['episodes']} | Time: {total_time:.0f}s ({total_time/3600:.1f}h)")

print(f"\nFixed Tasks:")
for t in EVAL_TASKS:
    b = baseline_scores.get(t, 0.0)
    a = final_scores.get(t, 0.0)
    status = "IMPROVED" if a > b + 0.01 else ("same" if abs(a-b) < 0.01 else "declined")
    print(f"  {t}: {b:.3f} -> {a:.3f} [{status}]")

gen_b = baseline_scores.get("generalization", 0.0)
gen_a = final_scores.get("generalization", 0.0)
print(f"\nGeneralization (unseen graphs): {gen_b:.3f} -> {gen_a:.3f}")

early_h = sum(episode_hint_rates[:20])/min(len(episode_hint_rates),20)
late_h = sum(episode_hint_rates[-20:])/min(len(episode_hint_rates),20)
print(f"\nHint win rate: {early_h:.0%} (early) -> {late_h:.0%} (late)")
if late_h < early_h:
    print("  Model surpassing heuristic -> genuine skill acquisition")

early_f = sum(episode_fuse_rates[:20])/min(len(episode_fuse_rates),20)
late_f = sum(episode_fuse_rates[-20:])/min(len(episode_fuse_rates),20)
early_r = sum(episode_retain_rates[:20])/min(len(episode_retain_rates),20)
late_r = sum(episode_retain_rates[-20:])/min(len(episode_retain_rates),20)
print(f"Fusion rate:    {early_f:.0%} -> {late_f:.0%}")
print(f"Retention rate: {early_r:.0%} -> {late_r:.0%}")

print(f"\nOutputs: {CONFIG['output_dir']}/")
print("=" * 70)

TRAINING SUMMARY
Episodes: 400 | Time: 10368s (2.9h)

Fixed Tasks:
  task1_chain: 0.084 -> 0.084 [same]
  task2_residual: -0.025 -> 0.015 [IMPROVED]
  task3_attention: 0.019 -> 0.034 [IMPROVED]
  task4_mixed: -0.485 -> 0.573 [IMPROVED]
  task5_adversarial: -0.726 -> 0.131 [IMPROVED]

Generalization (unseen graphs): -0.595 -> 0.252

Hint win rate: 65% (early) -> 0% (late)
  Model surpassing heuristic -> genuine skill acquisition
Fusion rate:    95% -> 100%
Retention rate: 88% -> 88%

Outputs: /workspace/scheduler-training-output/


In [ ]:
# Cell 20: Package results for download
import shutil
archive_name = os.path.join(WORKSPACE, "scheduler-training-results")
shutil.make_archive(archive_name, "zip", WORKSPACE, "scheduler-training-output")
subprocess.run(["tar", "-czf", archive_name + ".tar.gz", "-C", WORKSPACE, "scheduler-training-output"], check=True)
print(f"ZIP: {os.path.getsize(archive_name + '.zip') / 1e6:.1f} MB")
print(f"TAR: {os.path.getsize(archive_name + '.tar.gz') / 1e6:.1f} MB")
print("Download from file browser.")